# Deepfake Hybrid Detection — Colab Run
**Before running:** `Runtime → Change runtime type → T4 GPU`

Edit the two paths in **Step 1**, then run cells top to bottom.

- **Normal run:** Steps 1 → 2 → 3 → 4 (run_pipeline) → 5 (save)
- **If Step 4 fails mid-way:** use Step 4b (individual steps) to re-run only the failing phase

## Step 1 — Mount Drive & set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ═══════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit these values before running
# ═══════════════════════════════════════════════════════════════════════════

# Paths — datasets are used directly from Drive (no copy to local disk)
FFPP_IN_DRIVE = '/content/drive/MyDrive/ffpp'              # folder in Drive; set to None to download
CDF_IN_DRIVE  = '/content/drive/MyDrive/cdf/Celeb-DF-v2.zip'  # zip in Drive; set to None to download
CDF_DIR_DRIVE = '/content/drive/MyDrive/cdf/Celeb-DF-v2'   # pre-unzipped folder in Drive (set to None if not available)
PROJECT       = '/content/skripsi/deepfake_hybrid'
CONFIG        = f'{PROJECT}/config.yaml'
CDF_LOCAL     = '/content/celeb_df'  # local unzip destination (only used when CDF is not pre-unzipped in Drive)

# Dataset download (only used if Drive copy not found)
FFPP_NUM_REAL = None   # number of real videos to download (None = all ~1000)
FFPP_NUM_FAKE = None   # number of fake videos per method to download (None = all ~1000 each)
FFPP_SERVER   = 'EU2'  # EU, EU2, or CA — switch if download is slow

# Training — per-dataset sample size tiers
FFPP_SAMPLES_LIST = [100, 250, 500, 750]
CDF_SAMPLES_LIST  = [100, 250, 500, 750]
MAX_FRAMES    = 100    # max frames extracted per video
EPOCHS        = 30     # max epochs — early stopping (patience=5) handles actual duration
BATCH_SIZE    = 64     # T4 has 15 GB — 64 fills ~8 GB and gives ~4x GPU utilisation vs 16
NUM_WORKERS   = 4      # 4 keeps the GPU fed; raise to 6 if you still see GPU idle time
LR            = 1e-4
WEIGHT_DECAY  = 1e-4
IMAGE_SIZE    = 224
N_SEEDS       = 1      # increase to 3 for statistical validity
PRETRAINED    = True   # use ImageNet-pretrained XceptionNet

# Preprocessing
RECOMPUTE_FFT = True   # set to True to force recompute FFT cache (needed after changing fft_utils.py)

## Step 2 — Setup

_(command.txt §1)_

In [ ]:
import os
os.chdir('/')  # move to safe dir before deleting /content/skripsi

!rm -rf /content/skripsi
!git clone https://github.com/giovannyhalimko/skripsi.git /content/skripsi

!pip install --upgrade pip -q
!pip install -r {PROJECT}/requirements.txt -q
!apt-get install -qq ffmpeg

import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Device: {gpu_name}  ({vram_gb:.0f} GB)')

    # Auto-tune BATCH_SIZE and NUM_WORKERS based on GPU
    if 'A100' in gpu_name or vram_gb >= 35:
        BATCH_SIZE   = 128
        NUM_WORKERS  = 8
        COMPILE_MODEL = True   # torch.compile gives ~25% speedup on A100
        print('→ A100 detected: BATCH_SIZE=128, NUM_WORKERS=8, compile=True')
    elif vram_gb >= 14:        # T4 / V100 — 2 vCPUs, 15 GB VRAM
        BATCH_SIZE   = 64
        NUM_WORKERS  = 2       # T4 has 2 vCPUs; 4 workers triggers a warning
        COMPILE_MODEL = False
        print('→ T4/V100 detected: BATCH_SIZE=64, NUM_WORKERS=2, compile=False')
    else:                      # fallback / smaller GPU
        BATCH_SIZE   = 32
        NUM_WORKERS  = 2
        COMPILE_MODEL = False
        print('→ Small GPU detected: BATCH_SIZE=32, NUM_WORKERS=2, compile=False')
else:
    print('No GPU — running on CPU (very slow)')
    COMPILE_MODEL = False

## Step 3 — Copy datasets & patch config

In [ ]:
import yaml, os

# Patch config (paths will be set below after resolving FFPP_LOCAL)
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

cfg['datasets']['cdf']['real_keywords'] = ['real', 'authentic', 'youtube']
cfg['output_root'] = '/content/outputs'
cfg['epochs']      = EPOCHS
cfg['num_workers'] = NUM_WORKERS
cfg['batch_size']  = BATCH_SIZE

# ── FFPP: use Drive folder directly — no copy needed ──────────────────────
if FFPP_IN_DRIVE and os.path.isdir(FFPP_IN_DRIVE):
    FFPP_LOCAL = FFPP_IN_DRIVE
    print(f"FFPP found in Drive ({FFPP_IN_DRIVE}) — using directly (no copy).")
else:
    FFPP_LOCAL = '/content/ffpp'
    print("FFPP not found in Drive — downloading from FaceForensics server...")
    real_flag = f'--num-videos {FFPP_NUM_REAL}' if FFPP_NUM_REAL else ''
    fake_flag = f'--num-videos {FFPP_NUM_FAKE}' if FFPP_NUM_FAKE else ''

    !echo "" | python {PROJECT}/scripts/download_datasets.py --config {CONFIG} \
        --datasets original --compression c23 --type videos \
        {real_flag} --server {FFPP_SERVER}

    !echo "" | python {PROJECT}/scripts/download_datasets.py --config {CONFIG} \
        --datasets Deepfakes Face2Face FaceSwap NeuralTextures \
        --compression c23 --type videos \
        {fake_flag} --server {FFPP_SERVER}

    # Back up to Drive for future runs
    !cp -rn "{FFPP_LOCAL}/" "/content/drive/MyDrive/ffpp/"
    print("FFPP downloaded and backed up to Drive.")

cfg['datasets']['ffpp']['root'] = FFPP_LOCAL
with open(CONFIG, 'w') as f:
    yaml.dump(cfg, f)

!echo "FFPP real:" $(find {FFPP_LOCAL}/original_sequences -name "*.mp4" | wc -l) videos
!echo "FFPP fake:" $(find {FFPP_LOCAL}/manipulated_sequences -name "*.mp4" | wc -l) videos

os.chdir(PROJECT)

In [ ]:
# (FFPP Drive backup is now handled automatically in Step 3 above)

In [ ]:
import os

# ── CDF: use Drive directly if available ──────────────────────────────────
NEED_UNZIP = False
CDF_ZIP_SRC = None

if CDF_DIR_DRIVE and os.path.isdir(CDF_DIR_DRIVE):
    # Pre-unzipped folder in Drive — use it directly, no copy or unzip needed
    CDF_LOCAL = CDF_DIR_DRIVE
    print(f"Celeb-DF folder found in Drive ({CDF_DIR_DRIVE}) — using directly (no copy).")

elif CDF_IN_DRIVE and os.path.isfile(CDF_IN_DRIVE):
    # Zip in Drive — unzip directly from Drive to local (no copy step)
    print(f"Celeb-DF zip found in Drive ({CDF_IN_DRIVE}) — unzipping to local SSD...")
    CDF_ZIP_SRC = CDF_IN_DRIVE
    NEED_UNZIP = True

else:
    # Download zip to local, then unzip
    print("Celeb-DF not found in Drive — downloading via gdown...")
    !pip install -q gdown
    CDF_ZIP_SRC = '/content/celeb_df_tmp.zip'
    !gdown "1iLx76wsbi9itnkxSqz9BVBl4ZvnbIazj" -O "{CDF_ZIP_SRC}"
    # Back up zip to Drive for future runs
    import os as _os
    drive_cdf_dir = _os.path.dirname(CDF_IN_DRIVE) if CDF_IN_DRIVE else '/content/drive/MyDrive/cdf'
    _os.makedirs(drive_cdf_dir, exist_ok=True)
    dst_zip = CDF_IN_DRIVE if CDF_IN_DRIVE else f'{drive_cdf_dir}/Celeb-DF-v2.zip'
    !cp "{CDF_ZIP_SRC}" "{dst_zip}"
    print("Celeb-DF downloaded and backed up to Drive.")
    NEED_UNZIP = True

print("Celeb-DF source ready.")

In [ ]:
import shutil, subprocess, os
from pathlib import Path

_cdf_backup_proc = None  # background copy handle (waited on in Step 5)

if NEED_UNZIP:
    !mkdir -p {CDF_LOCAL}
    !unzip -q "{CDF_ZIP_SRC}" -d {CDF_LOCAL}

    root = Path(CDF_LOCAL)
    expected = {'Celeb-real', 'YouTube-real', 'Celeb-synthesis'}
    subdirs = [d for d in root.iterdir() if d.is_dir()]
    if subdirs and not expected.intersection({d.name for d in subdirs}):
        for item in subdirs[0].iterdir():
            shutil.move(str(item), str(root / item.name))
        subdirs[0].rmdir()
    print(f"CDF unzipped to {CDF_LOCAL}")

    # ── Back up unzipped folder to Drive in the background ────────────────
    # Future runs can set CDF_DIR_DRIVE to skip unzipping entirely.
    if CDF_DIR_DRIVE:
        os.makedirs(os.path.dirname(CDF_DIR_DRIVE), exist_ok=True)
        _cdf_backup_proc = subprocess.Popen(
            ['cp', '-r', CDF_LOCAL, CDF_DIR_DRIVE],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        print(f"Backing up unzipped CDF to Drive in background (PID {_cdf_backup_proc.pid})...")
        print(f"  → {CDF_DIR_DRIVE}")
        print("  (Next run: CDF_DIR_DRIVE will be used directly, no unzip needed)")
else:
    print(f"CDF using Drive directly: {CDF_LOCAL}")

!ls {CDF_LOCAL}

## Step 4 — Run everything

_(command.txt §2 — extract frames → splits → FFT cache → train → eval → tables)_

In [ ]:
import yaml, os
from datetime import datetime

# Write a fresh colab config with correct absolute paths
colab_cfg = {
    'output_root': '/content/outputs',
    'frame_sampling_fps': 5,
    'max_frames_per_video': MAX_FRAMES,
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'epochs': EPOCHS,
    'compile_model': COMPILE_MODEL,
    'fusion_mode': 'two_branch',
    'n_seeds': N_SEEDS,
    'datasets': {
        'ffpp': {
            'root': FFPP_LOCAL,
            'real_keywords': ['original', 'real', 'pristine', 'actors', 'youtube'],
            'fake_keywords': ['fake', 'manipulated', 'deepfakes', 'faceswap',
                              'neuraltextures', 'deepfakedetection', 'faceshifter', 'face2face'],
        },
        'cdf': {
            'root': CDF_LOCAL,
            'real_keywords': ['real', 'authentic', 'youtube'],
            'fake_keywords': ['fake', 'synthesis', 'deepfake'],
        }
    }
}

COLAB_CONFIG = f'{PROJECT}/colab_config.yaml'
with open(COLAB_CONFIG, 'w') as f:
    yaml.dump(colab_cfg, f, default_flow_style=False)

with open(COLAB_CONFIG) as f:
    check = yaml.safe_load(f)
print("FFPP root:", check['datasets']['ffpp']['root'])
print("CDF  root:", check['datasets']['cdf']['root'])
print("compile_model:", check['compile_model'])
assert check['datasets']['ffpp']['root'] == FFPP_LOCAL, "FFPP path wrong!"

os.chdir(PROJECT)
pretrained_flag = '--pretrained' if PRETRAINED else ''
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
drive_base = f'/content/drive/MyDrive/skripsi_outputs/{ts}'

# Both lists must be the same length for paired iteration
assert len(FFPP_SAMPLES_LIST) == len(CDF_SAMPLES_LIST), "Sample lists must have the same length"

ffpp_preprocessed = False
cdf_preprocessed = False

for i, (ffpp_n, cdf_n) in enumerate(zip(FFPP_SAMPLES_LIST, CDF_SAMPLES_LIST)):
    print(f"\n{'#' * 60}")
    print(f"  Round {i+1}/{len(FFPP_SAMPLES_LIST)}: FFPP n={ffpp_n} → CDF n={cdf_n} → cross-eval")
    print(f"{'#' * 60}\n")

    # ── Step A: Train FFPP ────────────────────────────────────────────────
    print(f"--- FFPP n={ffpp_n} ---")
    force_fft_flag = '--force-fft' if (RECOMPUTE_FFT and not ffpp_preprocessed) else ''
    skip_flag = '--skip-preprocess' if ffpp_preprocessed else ''
    !python scripts/run_pipeline.py \
        --config colab_config.yaml \
        --dataset FFPP \
        --n-samples {ffpp_n} \
        --max-frames {MAX_FRAMES} \
        --epochs {EPOCHS} \
        {force_fft_flag} \
        {skip_flag} \
        {pretrained_flag}
    ffpp_preprocessed = True
    dst = f'{drive_base}/FFPP/n{ffpp_n}'
    !mkdir -p "{dst}"
    !cp -r /content/outputs/tables/n{ffpp_n} "{dst}/tables" 2>/dev/null; true
    !cp -r /content/outputs/runs/*_FFPP_n{ffpp_n}_* "{dst}/runs/" 2>/dev/null; true

    # ── Step B: Train CDF ─────────────────────────────────────────────────
    print(f"--- CDF n={cdf_n} ---")
    force_fft_flag = '--force-fft' if (RECOMPUTE_FFT and not cdf_preprocessed) else ''
    skip_flag = '--skip-preprocess' if cdf_preprocessed else ''
    !python scripts/run_pipeline.py \
        --config colab_config.yaml \
        --dataset CDF \
        --n-samples {cdf_n} \
        --max-frames {MAX_FRAMES} \
        --epochs {EPOCHS} \
        {force_fft_flag} \
        {skip_flag} \
        {pretrained_flag}
    cdf_preprocessed = True
    dst = f'{drive_base}/CDF/n{cdf_n}'
    !mkdir -p "{dst}"
    !cp -r /content/outputs/tables/n{cdf_n} "{dst}/tables" 2>/dev/null; true
    !cp -r /content/outputs/runs/*_CDF_n{cdf_n}_* "{dst}/runs/" 2>/dev/null; true

    # ── Step C: Cross-dataset eval ────────────────────────────────────────
    # Both manifests now exist — fill FFPP→CDF and CDF→FFPP in tables
    print(f"--- Cross-eval n={ffpp_n}/{cdf_n} ---")
    for n in sorted(set([ffpp_n, cdf_n])):
        !python scripts/run_all.py \
            --config colab_config.yaml \
            --dataset both \
            --n-samples {n} \
            {pretrained_flag}
        dst = f'{drive_base}/cross/n{n}'
        !mkdir -p "{dst}"
        !cp -r /content/outputs/tables/n{n} "{dst}/tables" 2>/dev/null; true
        print(f"  Saved cross tables n={n} to Drive")

# ── Plots ─────────────────────────────────────────────────────────────────
all_n = sorted(set(FFPP_SAMPLES_LIST + CDF_SAMPLES_LIST))
n_samples_str = ','.join(str(n) for n in all_n)
!python scripts/plot_results.py --config colab_config.yaml --n-samples {n_samples_str}
!cp -r /content/outputs/plots "{drive_base}/plots" 2>/dev/null; true

print(f"\n{'#' * 60}")
print(f"  ALL DONE — FFPP={FFPP_SAMPLES_LIST}, CDF={CDF_SAMPLES_LIST}")
print(f"{'#' * 60}")

---
## Step 4b — Individual steps (only if Step 4 fails)

_(command.txt §3)_

Run only the cell for the phase that failed — skip the ones that already completed.

In [ ]:
# Extract frames
!python scripts/extract_frames.py --config {CONFIG} --dataset FFPP --fps 5 --max-frames {MAX_FRAMES} --n-samples {N_SAMPLES}
!python scripts/extract_frames.py --config {CONFIG} --dataset CDF  --fps 5 --max-frames {MAX_FRAMES} --n-samples {N_SAMPLES}

In [ ]:
# Build splits
!python scripts/build_splits.py --config {CONFIG} --dataset FFPP
!python scripts/build_splits.py --config {CONFIG} --dataset CDF

In [ ]:
# FFT cache
force_flag = '--force' if RECOMPUTE_FFT else ''
!python scripts/compute_fft_cache.py --config {CONFIG} --dataset FFPP --num-workers {NUM_WORKERS} {force_flag}
!python scripts/compute_fft_cache.py --config {CONFIG} --dataset CDF  --num-workers {NUM_WORKERS} {force_flag}

In [ ]:
# Train individual models
pretrained_flag = '--pretrained' if PRETRAINED else ''
!python scripts/train.py --config {CONFIG} --dataset FFPP --model spatial {pretrained_flag}
!python scripts/train.py --config {CONFIG} --dataset FFPP --model freq
!python scripts/train.py --config {CONFIG} --dataset FFPP --model hybrid  {pretrained_flag}

In [ ]:
# Full matrix: train + eval + tables
pretrained_flag = '--pretrained' if PRETRAINED else ''
!python scripts/run_all.py --config {CONFIG} --n-samples {N_SAMPLES} {pretrained_flag}

---
## Step 5 — Save outputs to Drive

**Run before closing the session.**

Output locations (command.txt §Output locations):
- `outputs/tables/Table1_in_dataset.csv`, `Table2_cross_dataset.csv`, `Table3_generalization_drop.csv`
- `outputs/runs/<model>_<dataset>_n<N>_seed<N>/best.pt`
- `outputs/runs/<model>_<dataset>_n<N>_seed<N>/train.log`

In [ ]:
from datetime import datetime

# ── Wait for background CDF Drive backup to finish ────────────────────────
if '_cdf_backup_proc' in dir() and _cdf_backup_proc is not None:
    print("Waiting for CDF Drive backup to finish...")
    _cdf_backup_proc.wait()
    rc = _cdf_backup_proc.returncode
    if rc == 0:
        print(f"CDF backup complete → {CDF_DIR_DRIVE}")
    else:
        print(f"CDF backup finished with errors (return code {rc}). Check Drive manually.")

# ── Save training outputs ──────────────────────────────────────────────────
ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'/content/drive/MyDrive/skripsi_outputs/{ts}'

!cp -r /content/outputs "{dst}"
!echo "Saved to: {dst}"
!ls "{dst}"

---
## Step 6 — Download outputs to local machine

Zips tables, logs, and conclusion files (no checkpoints) and triggers a browser download.

In [ ]:
import os, zipfile
from datetime import datetime
from google.colab import files

OUTPUT_ROOT = '/content/outputs'
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_path = f'/content/skripsi_results_{ts}.zip'

# Zip tables, train logs, and conclusion files — skip large .pt checkpoints
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, filenames in os.walk(OUTPUT_ROOT):
        # Skip runs/ subdirectories that only contain checkpoints
        dirs[:] = [d for d in dirs if d != 'frames' and d != 'fft_cache']
        for fname in filenames:
            if fname.endswith('.pt'):
                continue  # skip model checkpoints (~90MB each)
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, '/content')
            zf.write(fpath, arcname)

size_mb = os.path.getsize(zip_path) / 1e6
print(f'Zip ready: {zip_path}  ({size_mb:.1f} MB)')
print('Starting download...')
files.download(zip_path)